# Notebook 02 — Category Classifier (3-Model Comparison)

**Team ARAJ · Phase 3 ML · Owner: Arpan, Ashfaaq**

Predicts the **spend category** of a receipt from its text (merchant + items).

- **Data:** the cleaned, pre-processed splits `category_{train,val,test}.csv` (leakage-free — `source` is *not* a feature).
- **Target:** `category_3class` — *Supermarket / Grocery*, *Food & Beverage*, *General Retail*.
  (We deliberately avoid the 2-class `category` label, which is 1:1 with the dataset source = leakage.)
- **As Prof. Uma asked:** train **3 different models**, compare them, pick the best, and justify the choice.
- **Metric:** classes are imbalanced (671 / 274 / 53), so we rank by **macro-F1** and use `class_weight='balanced'`.

## Step 1 — Imports

In [2]:
# sklearn for TF-IDF + 3 candidate classifiers; joblib to save the winner
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
import joblib

RANDOM_STATE = 42
DATA = Path('../dataset/processed')
MODELS = Path('../ml-service/models')
MODELS.mkdir(parents=True, exist_ok=True)

## Step 2 — Load the pre-processed splits

These were produced by `dataset/prepare_dataset.py` (stratified 70/15/15). We do **not** re-split here.

In [11]:
train = pd.read_csv(DATA / 'category_train.csv')
val   = pd.read_csv(DATA / 'category_val.csv')
test  = pd.read_csv(DATA / 'category_test.csv')

TARGET = 'category_3class'   # 3-class spend category (leakage-free target)
FEATURE = 'text'             # merchant + items, prepared upstream; 'source' is excluded

for name, df in [('train', train), ('val', val), ('test', test)]:
    print(f'{name:5s}: {len(df):4d} rows | class balance: {df[TARGET].value_counts().to_dict()}')

train:  998 rows | class balance: {'General Retail': 671, 'Food & Beverage': 274, 'Supermarket / Grocery': 53}
val  :  213 rows | class balance: {'General Retail': 148, 'Food & Beverage': 54, 'Supermarket / Grocery': 11}
test :  216 rows | class balance: {'General Retail': 136, 'Food & Beverage': 69, 'Supermarket / Grocery': 11}


## Step 3 — Features & labels

In [4]:
X_train, y_train = train[FEATURE].fillna(''), train[TARGET]
X_val,   y_val   = val[FEATURE].fillna(''),   val[TARGET]
X_test,  y_test  = test[FEATURE].fillna(''),  test[TARGET]

# One shared TF-IDF vectoriser, fit on TRAIN only (no leakage from val/test).
# We save this vectoriser + the winning classifier separately so ml-service/classifier.py
# can load models/tfidf.pkl and models/classifier.pkl.
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2)
Xtr = vectorizer.fit_transform(X_train)
Xva = vectorizer.transform(X_val)
Xte = vectorizer.transform(X_test)
print('TF-IDF vocabulary size:', len(vectorizer.vocabulary_))

TF-IDF vocabulary size: 1157


## Step 4 — Define the 3 candidate models

| Model | Why consider it |
|---|---|
| Logistic Regression | Strong, fast linear baseline for sparse text |
| Linear SVM | Often the best performer on TF-IDF features |
| Random Forest | Non-linear, robust to noisy features |

All use `class_weight='balanced'` to handle the class imbalance.

In [5]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced',
                                               random_state=RANDOM_STATE),
    'Linear SVM':          LinearSVC(class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                   n_jobs=-1, random_state=RANDOM_STATE),
}

## Step 5 — Train each model & evaluate on the validation set

In [6]:
results = []
trained = {}
for name, model in models.items():
    model.fit(Xtr, y_train)
    trained[name] = model
    pred = model.predict(Xva)
    results.append({
        'model': name,
        'val_accuracy': round(accuracy_score(y_val, pred), 4),
        'val_macro_f1': round(f1_score(y_val, pred, average='macro'), 4),
        'val_weighted_f1': round(f1_score(y_val, pred, average='weighted'), 4),
    })
    print(f'\n=== {name} (validation) ===')
    print(classification_report(y_val, pred, zero_division=0))


=== Logistic Regression (validation) ===
                       precision    recall  f1-score   support

      Food & Beverage       0.88      0.91      0.89        54
       General Retail       0.96      0.93      0.95       148
Supermarket / Grocery       0.77      0.91      0.83        11

             accuracy                           0.92       213
            macro avg       0.87      0.92      0.89       213
         weighted avg       0.93      0.92      0.93       213


=== Linear SVM (validation) ===
                       precision    recall  f1-score   support

      Food & Beverage       0.94      0.89      0.91        54
       General Retail       0.96      0.96      0.96       148
Supermarket / Grocery       0.71      0.91      0.80        11

             accuracy                           0.94       213
            macro avg       0.87      0.92      0.89       213
         weighted avg       0.94      0.94      0.94       213


=== Random Forest (validation) ===
 

## Step 6 — Comparison table & pick the best (by macro-F1)

In [7]:
comparison = pd.DataFrame(results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
print(comparison.to_string(index=False))

best_name = comparison.iloc[0]['model']
best_model = trained[best_name]
print(f'\n>>> Best model by macro-F1: {best_name}')

              model  val_accuracy  val_macro_f1  val_weighted_f1
      Random Forest        0.9484        0.9092           0.9484
         Linear SVM        0.9390        0.8912           0.9398
Logistic Regression        0.9249        0.8898           0.9257

>>> Best model by macro-F1: Random Forest


## Step 7 — Confusion matrix for the best model (validation)

Printed as text so the notebook runs without matplotlib; the plot is attempted if matplotlib is installed.

In [8]:
labels_sorted = sorted(y_train.unique())
val_pred = best_model.predict(Xva)
cm = confusion_matrix(y_val, val_pred, labels=labels_sorted)
print('Labels:', labels_sorted)
print(pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted))

try:
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    disp = ConfusionMatrixDisplay(cm, display_labels=labels_sorted)
    disp.plot(cmap='Blues', xticks_rotation=45)
    plt.title(f'{best_name} — validation confusion matrix')
    plt.tight_layout(); plt.show()
except ImportError:
    print('\n(matplotlib not installed — run: pip install matplotlib — to see the plotted matrix)')

Labels: ['Food & Beverage', 'General Retail', 'Supermarket / Grocery']
                       Food & Beverage  General Retail  Supermarket / Grocery
Food & Beverage                     47               7                      0
General Retail                       0             145                      3
Supermarket / Grocery                0               1                     10

(matplotlib not installed — run: pip install matplotlib — to see the plotted matrix)


## Step 8 — Final check on the held-out test set

Only the winning model is evaluated on test — this is the number we report.

In [9]:
test_pred = best_model.predict(Xte)
print(f'=== {best_name} (TEST) ===')
print(classification_report(y_test, test_pred, zero_division=0))
print('Test macro-F1   :', round(f1_score(y_test, test_pred, average='macro'), 4))
print('Test accuracy   :', round(accuracy_score(y_test, test_pred), 4))

=== Random Forest (TEST) ===
                       precision    recall  f1-score   support

      Food & Beverage       0.95      0.87      0.91        69
       General Retail       0.94      0.98      0.96       136
Supermarket / Grocery       0.92      1.00      0.96        11

             accuracy                           0.94       216
            macro avg       0.94      0.95      0.94       216
         weighted avg       0.94      0.94      0.94       216

Test macro-F1   : 0.942
Test accuracy   : 0.9444


## Step 9 — Save the winning model + vectoriser

Saved as `classifier.pkl` + `tfidf.pkl` so `ml-service/classifier.py` can load them for live inference.

In [10]:
joblib.dump(best_model, MODELS / 'classifier.pkl')
joblib.dump(vectorizer, MODELS / 'tfidf.pkl')
print('Saved:')
print('  ', MODELS / 'classifier.pkl')
print('  ', MODELS / 'tfidf.pkl')
print(f'Winner: {best_name}')

Saved:
   ../ml-service/models/classifier.pkl
   ../ml-service/models/tfidf.pkl
Winner: Random Forest


## Step 10 — Justification

> **Chosen model: Random Forest** (300 trees, `class_weight='balanced'`).
> **Why:** it took the highest **validation macro-F1 — 0.9092**, against 0.8912 for Linear SVM and 0.8898 for
> Logistic Regression (Step 6). Macro-F1 is the deciding metric because the classes are heavily imbalanced
> (671 / 274 / 53 in train): a model that ignored *Supermarket / Grocery* entirely could still post ~0.90
> plain accuracy, and macro-F1 refuses to reward that.

The margin is real but modest — **0.018 macro-F1 over Linear SVM**. Random Forest earns it mainly on
precision for *Food & Beverage* (1.00 vs 0.94 on validation), i.e. when it says "restaurant" it is right.
It was also chosen on validation only; test was touched once, after the decision.

### Test-set result (the reported number)

**Macro-F1 0.942 · accuracy 0.9444** on 216 held-out receipts, against a 0.80 target.

| Class | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Food & Beverage | 0.95 | **0.87** | 0.91 | 69 |
| General Retail | 0.94 | 0.98 | 0.96 | 136 |
| Supermarket / Grocery | 0.92 | 1.00 | 0.96 | 11 |

### Read this honestly — two caveats

**1. The test score is *higher* than validation (0.942 vs 0.909), which is not the usual direction.**
It is driven almost entirely by *Supermarket / Grocery* jumping from F1 0.83 on validation to 0.96 on test —
on a class with **11 samples in each split**. One or two receipts moving flips that figure by ~0.08. The
grocery column should be treated as indicative, not precise, and the honest headline is "≈0.94 macro-F1 with
a small-class caveat" rather than a hard 0.942.

**2. The genuine weak spot is *Food & Beverage* recall — 0.87.**
Nine of 69 restaurant receipts are missed, and the validation confusion matrix shows where they go: 7 of them
land in *General Retail*, never in *Grocery*. These are the terse receipts — a café bill with two line items
and no dish names has little for TF-IDF to grip (the whole vocabulary is only 1,157 terms after `min_df=2`).

### Known limitations

- **The label itself is a heuristic.** `category_3class` comes from a keyword mapping in pre-processing, so the
  achievable ceiling is the quality of that mapping, not of the classifier. A model cannot beat its labels.
- **Only text is used.** Date and merchant were excluded because they are missing for 50.7% of rows, and
  structurally so — CORD never recorded them. Imputing would be inventing data.
- **Corpus is Malaysian and Indonesian**, so Indian merchant vocabulary is under-represented relative to where
  the system is actually aimed.

### Next steps, in priority order

1. **Evaluate against the 11 real Indian spend categories** in `indian_category_folders.csv` — human-assigned
   labels the model has never seen. This is the strongest available check on the heuristic-label ceiling, and
   the data already exists.
2. **Collect more Grocery receipts.** At 53 training samples it is the only class whose score cannot be trusted.
3. **Revisit the terse-receipt failure** — character n-grams or a merchant-name lexicon would help where item
   text is nearly empty.
